In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
try:
    from Preprocessing.analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
except ModuleNotFoundError:
    from analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
from hdmf_zarr.nwb import NWBZarrIO
import matplotlib.pyplot as plt
import anndata as ad
import pickle
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from pathlib import Path
import os
import sys
import time
import zarr
import numcodecs
import shutil
import re
RESAMPLE_RATE = 10    # Hz
PRE_STIM      = 1.0   # seconds before stimulus onset
POST_STIM     = 1.0   # seconds after stimulus offset

SCRATCH_DIR = Path("/root/capsule/scratch")

DATA_DIR = Path("/root/capsule/data")
STIM_ALIGNED_DIR = SCRATCH_DIR / "ophys" / "stim_aligned"
MORPH_DIR = SCRATCH_DIR / "morphology"
TRANSCRIPT_DIR = SCRATCH_DIR / 'transcriptomics'


MULTIMODAL = SCRATCH_DIR / "multimodal_data"
MULTIMODAL.mkdir(parents=True, exist_ok=True)

# Import tuning and GLM functions from code/functions/
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('.'), '..')))
from functions.tuning import compute_tuning_for_session, save_tuning_to_zarr
from functions.glm import add_glm_aggregate_columns

load data information

In [ ]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [778174, 786297, 797371]


In [ ]:
CHECKPOINT_INTERVAL = 5 * 60  # seconds between timed checkpoints

for mouse_id in mouse_ids:
    in_path = STIM_ALIGNED / f"{mouse_id}_multimodal_data.pkl"
    with open(in_path, 'rb') as f:
            session_data = pickle.load(f)
    for session_name in ['session_1', 'session_2', 'session_3']:
        
        out_path = STIM_ALIGNED / f"{mouse_id}_{session_name}_correlation_matrix.pkl"

        # ── Load existing partial results if available ──────────────
        if out_path.exists():
            with open(out_path, 'rb') as f:
                corr_results = pickle.load(f)
            print(f"[resume] Loaded partial results from {out_path.name}")
        else:
            corr_results = {}

        

        dirty = False  # track whether we need to save
        last_save_time = time.time()

        def _timed_checkpoint(force=False):
            """Save corr_results if ≥ CHECKPOINT_INTERVAL seconds elapsed (or force=True)."""
            global last_save_time, dirty
            now = time.time()
            if force or (now - last_save_time >= CHECKPOINT_INTERVAL):
                with open(out_path, 'wb') as f:
                    pickle.dump(corr_results, f, protocol=pickle.HIGHEST_PROTOCOL)
                elapsed = int(now - last_save_time)
                print(f"  [checkpoint @ {elapsed}s] saved {out_path.name}")
                last_save_time = now
                dirty = False

        for stim_name in session_data['ophys']['drifting_gratings'][session_name]:
            if stim_name not in corr_results:
                corr_results[stim_name] = {}
            dff = session_data['ophys']['drifting_gratings'][session_name][stim_name]['dff']
            n_cells: int = dff.shape[2]

            if stim_name == "GratingStim":
                # continue
                trial_info = session_data['ophys']['drifting_gratings'][session_name][stim_name]['trial_info']
                combinations = trial_info.drop_duplicates(['stim_name', 'contrast','temporal_frequency','orientation']).\
                    sort_values(['stim_name', 'contrast','temporal_frequency','orientation'])[['stim_name', 'contrast','temporal_frequency','orientation']].reset_index(drop = True)
                for combination in combinations.itertuples(index=False):
                    combo_key = f"{combination.stim_name.replace('drifting_gratings','sweeping')}_{int(combination.contrast*100)}%_{int(combination.temporal_frequency)}Hz_{int(combination.orientation)}°"

                    # Check for fully computed or partial result
                    start_row = 0
                    if combo_key in corr_results[stim_name]:
                        existing = corr_results[stim_name][combo_key]
                        if isinstance(existing, dict):
                            # Partial result: {'matrix': ..., 'completed_rows': N}
                            start_row = existing['completed_rows']
                            combo_corr = existing['matrix']
                            print(f"[resume] {combo_key} from row {start_row}/{n_cells}")
                        elif isinstance(existing, np.ndarray) and existing.shape == (n_cells, n_cells):
                            print(f"[skip] {combo_key} already computed")
                            continue
                    else:
                        combo_corr = np.zeros((n_cells, n_cells))

                    stim_inds = trial_info[
                        (trial_info['stim_name'] == combination.stim_name) &
                        (trial_info['contrast'] == combination.contrast) &
                        (trial_info['temporal_frequency'] == combination.temporal_frequency) &
                        (trial_info['orientation'] == combination.orientation)
                    ].index.values[::2].astype(int)

                    print(f"Computing {combo_key} ({n_cells} cells, from row {start_row}) ...", end=" ", flush=True)

                    for i in range(start_row, n_cells):
                        for j in range(n_cells):
                            combo_corr[i, j] = np.nanmean([np.corrcoef(dff[int(si), 10:-10, i], dff[int(si), 10:-10, j])[0, 1] for si in stim_inds])
                            if np.isnan(combo_corr[i, j]):
                                print(combo_key, i, j, 'nan')

                        # Timed partial checkpoint (save progress per-row)
                        if time.time() - last_save_time >= CHECKPOINT_INTERVAL:
                            corr_results[stim_name][combo_key] = {'matrix': combo_corr, 'completed_rows': i + 1}
                            dirty = True
                            _timed_checkpoint(force=True)

                    # Store final complete matrix (plain ndarray, not dict)
                    corr_results[stim_name][combo_key] = combo_corr
                    dirty = True
                    print("done")
                    _timed_checkpoint(force=True)

            # ── "all" correlation matrix ────────────────────────────
            start_row = 0
            if 'all' in corr_results[stim_name]:
                existing = corr_results[stim_name]['all']
                if isinstance(existing, dict):
                    start_row = existing['completed_rows']
                    correlation_matrix = existing['matrix']
                    print(f"[resume] {stim_name}/all from row {start_row}/{n_cells}")
                elif isinstance(existing, np.ndarray) and existing.shape == (n_cells, n_cells):
                    print(f"[skip] {stim_name}/all already computed")
                    continue
            else:
                correlation_matrix = np.zeros((n_cells, n_cells))

            print(f"Computing {stim_name}/all ({n_cells} cells, from row {start_row}) ...", end=" ", flush=True)
            n_trials = dff.shape[0]
            for i in range(start_row, n_cells):
                for j in range(n_cells):
                    correlation_matrix[i, j] = np.nanmean([np.corrcoef(dff[int(stim_ind), 10:-10, i], dff[int(stim_ind), 10:-10, j])[0, 1] for stim_ind in range(n_trials)])

                # Timed partial checkpoint
                if time.time() - last_save_time >= CHECKPOINT_INTERVAL:
                    corr_results[stim_name]['all'] = {'matrix': correlation_matrix, 'completed_rows': i + 1}
                    dirty = True
                    _timed_checkpoint(force=True)

            corr_results[stim_name]['all'] = correlation_matrix
            dirty = True
            print("done")
            _timed_checkpoint(force=True)

        if dirty:
            _timed_checkpoint(force=True)
            print(f"✓ {out_path.name} (complete)")
        else:
            print(f"✓ {out_path.name} (nothing new to compute)")

[resume] Loaded partial results from 778174_session_1_correlation_matrix.pkl
[skip] GreyScreen/all already computed
[skip] sweeping_TF_80%_1Hz_0° already computed
[skip] sweeping_TF_80%_1Hz_45° already computed
[skip] sweeping_TF_80%_1Hz_90° already computed
[skip] sweeping_TF_80%_1Hz_135° already computed
[skip] sweeping_TF_80%_1Hz_180° already computed
[skip] sweeping_TF_80%_1Hz_225° already computed
[skip] sweeping_TF_80%_1Hz_270° already computed
[skip] sweeping_TF_80%_1Hz_315° already computed
[skip] sweeping_TF_80%_2Hz_0° already computed
[skip] sweeping_TF_80%_2Hz_45° already computed
[skip] sweeping_TF_80%_2Hz_90° already computed
[skip] sweeping_TF_80%_2Hz_135° already computed
[skip] sweeping_TF_80%_2Hz_180° already computed
[skip] sweeping_TF_80%_2Hz_225° already computed
[skip] sweeping_TF_80%_2Hz_270° already computed
[skip] sweeping_TF_80%_2Hz_315° already computed
[skip] sweeping_TF_80%_4Hz_0° already computed
[skip] sweeping_TF_80%_4Hz_45° already computed
[skip] sweepi